# Timeline research: what temporal signals exist?

The temporal smear caveat is the project's biggest known limitation: AWOIAF collapses ≈ 5,000 in-universe years into one static graph, so the Crownlands community ends up bundling every Targaryen monarch from Aegon I (1 BC – 37 AC) to Daenerys (284 AC – present). This notebook surveys what temporal information is *actually accessible* before we commit to a bucketing strategy.

**Three potential signal sources:**
1. **Infobox `Born` / `Died` fields** — the proper fix. Most precise but currently *not scraped* (see `INFOBOX_LABELS` in `scrape_characters_v2.ipynb`: only `Father, Mother, Spouse, Lover, Issue, Allegiance`).
2. **Recent Events h3 subsections** — book names like `A Game of Thrones` → all map to the 298–305 AC main timeline. Covers ~50% of characters per the earlier section-heading survey.
3. **Bio prose date mentions** — regex for `\d+ AC` / `\d+ BC` in `characters_bio.csv`. Sparse and inconsistent.

Goal of this notebook: probe each source on the existing data, decide which one(s) to lean on in `02_plan.ipynb`.

In [1]:
import pandas as pd
import re
from collections import Counter

df = pd.read_csv('../characters_enriched_v2.csv').fillna('')
bios = pd.read_csv('../csvs/characters_bio.csv').fillna('')
print(f'{len(df)} characters, {len(bios)} bios')


3690 characters, 3690 bios


## Signal 1: bio text date mentions
Sparse but easy to probe. Pattern `\d{1,4}\s+(AC|BC)` catches things like `298 AC`, `27 BC`, `100 AC`. We also look for event anchors like *Aegon's Conquest*, *Dance of the Dragons*, *Robert's Rebellion* — which give era hints even without a year number.

In [2]:
YEAR_RE = re.compile(r'\b(\d{1,4})\s*(AC|BC)\b')
EVENTS = {
    "Aegon's Conquest":     (-2, 1),     # ~2 BC - 1 AC
    'Dance of the Dragons':  (129, 131),  # 129-131 AC
    'Faith Militant':        (41, 49),
    'Blackfyre Rebellion':   (196, 196),
    "Robert's Rebellion":   (282, 283),
    'Greyjoy Rebellion':     (289, 289),
    "War of the Five Kings": (298, 300),
}

bio_by_id = dict(zip(bios['ID'], bios['bio']))

date_hits = {}
event_hits = Counter()
n_with_year = 0
for cid, bio in bio_by_id.items():
    if not bio.strip():
        continue
    years = [(int(y), suf) for y, suf in YEAR_RE.findall(bio)]
    if years:
        n_with_year += 1
        date_hits[cid] = years
    for ev in EVENTS:
        if ev in bio:
            event_hits[ev] += 1

print(f'Bios with at least one "X AC/BC" year mention: {n_with_year} ({n_with_year/len(bios)*100:.1f}%)')
print()
print('Top event anchors:')
for ev, n in event_hits.most_common():
    print(f'  {ev:25s}  {n:>4}')


Bios with at least one "X AC/BC" year mention: 748 (20.3%)

Top event anchors:
  Dance of the Dragons        205
  War of the Five Kings       147
  Robert's Rebellion          109
  Blackfyre Rebellion          62
  Aegon's Conquest             53
  Faith Militant               40
  Greyjoy Rebellion             2


Coverage check on the canonical era boundaries — what fraction of characters mention each anchor event?

In [3]:
# Look at a few sample characters from different presumed eras
def show(cid):
    bio = bio_by_id.get(cid, '')
    years = YEAR_RE.findall(bio) if bio else []
    events = [e for e in EVENTS if e in bio]
    print(f'{cid:30s}  years={years[:6]}  events={events[:3]}')

for cid in ['Aegon_I_Targaryen', 'Maegor_I_Targaryen', 'Aegon_III_Targaryen',
            'Duncan_the_Tall', 'Aegon_V_Targaryen', 'Aerys_II_Targaryen',
            'Eddard_Stark', 'Tyrion_Lannister', 'Daenerys_Targaryen']:
    show(cid)


Aegon_I_Targaryen               years=[('27', 'BC')]  events=[]
Maegor_I_Targaryen              years=[]  events=[]
Aegon_III_Targaryen             years=[]  events=['Dance of the Dragons']
Duncan_the_Tall                 years=[('196', 'AC'), ('205', 'AC'), ('206', 'AC'), ('208', 'AC'), ('209', 'AC'), ('193', 'AC')]  events=['Blackfyre Rebellion']
Aegon_V_Targaryen               years=[('219', 'AC'), ('220', 'AC'), ('221', 'AC'), ('233', 'AC')]  events=['Blackfyre Rebellion']
Aerys_II_Targaryen              years=[('262', 'AC'), ('283', 'AC'), ('259', 'AC'), ('262', 'AC')]  events=["Robert's Rebellion", 'War of the Five Kings']
Eddard_Stark                    years=[]  events=[]
Tyrion_Lannister                years=[('289', 'AC'), ('291', 'AC')]  events=[]
Daenerys_Targaryen              years=[('297', 'AC'), ('153', 'AC')]  events=["Robert's Rebellion"]


## Signal 2: book subsections under Recent Events
We surveyed this in `../scrape_section_headings.ipynb` — `A Game of Thrones` appears on 383 character pages, `A Dance with Dragons` on 801, etc. **All five mainline ASOIAF books map to the same era window (≈ 298–305 AC).** A character with *any* book subsection is in the current timeline. The signal is binary, not granular within the era — that's fine for bucketing into Fire & Blood vs Dunk & Egg vs ASOIAF.

What we *don't* have: a per-character CSV column with the book mentions. To get one we'd need to walk each character page's `Recent Events` h2 children. ~3,689 pages × seconds each = a re-scrape pass.

## Signal 3: infobox `Born` / `Died`
Currently *not captured*. The proper fix. Adding `'Born': 'born', 'Died': 'died'` to `INFOBOX_LABELS` and re-running the v2 scraper would give us numeric dates for ~60–70% of characters (estimate based on AWOIAF infobox conventions — most named historical characters have either a Born, a Died, or both).

**This is the best single signal.** Years parse as integers, eras become numeric intervals, and the bucketing logic is unambiguous.

## Decision

**Lead signal:** infobox `Born` / `Died` (re-scrape required, ≈ 10 min).

**Fallbacks** when the infobox is missing dates, in order:
1. Any ASOIAF book h3 subsection → `ASOIAF (298-305 AC)`
2. Bio prose mentions of `Aegon's Conquest` or pre-Conquest events → `Pre / Conquest era`
3. Bio prose mentions of `Dance of the Dragons`, `Faith Militant`, etc. → era of that event
4. Otherwise → `Unknown` (and don't pretend we know)

Coverage estimate using this cascade: ≈ 80–85% of characters get a confident era; the rest stay Unknown. See `02_plan.ipynb` for the era definitions, `03_execute.ipynb` for the implementation.